In [1]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from streamlit_autorefresh import st_autorefresh
import datetime


2026-03-10 14:47:51.394 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [2]:
def get_engine():
    user = "wconceicao"
    password = "zJm7$j%qRU@WoCxM"
    host = "dw-ro.data.grupoa.education"
    port = "5432"
    database = "postgres"
    password_encoded = quote_plus(password)
    return create_engine(f"postgresql+psycopg2://{user}:{password_encoded}@{host}:{port}/{database}")

# --- QUERY ATUALIZADA (Focada em Áreas) ---
QRY_VENDAS = """
WITH Sispag AS (
    SELECT 
        vd_produto.nomeresumido,
        TO_CHAR(DATE_TRUNC('hour', vd_compra.datahora), 'HH24:MI') AS hr,
        CASE 
            WHEN programs.area ~* 'Mental' THEN 'Psicologia'
            WHEN programs.area IS NOT NULL THEN programs.area
            ELSE CASE 
                WHEN vd_produto.nomeresumido::text = ANY(ARRAY['PROURGEM','PROENDÓCRINO','PRO-UROLOGIA','PROTERAPÊUTICA']) THEN 'Medicina'
                WHEN vd_produto.nomeresumido::text = ANY(ARRAY['PROEMED','PRORN','PRONEUROPED']) THEN 'Pediatria'
                WHEN vd_produto.nomeresumido::text = ANY(ARRAY['PROPSCOMED','PROPSIQ']) THEN 'Psiquiatria'
                WHEN vd_produto.nomeresumido::text = 'PROMEVET' THEN 'Veterinária'
                WHEN vd_produto.nomeresumido::text = 'PRONUTRI' THEN 'Nutrição'
                WHEN vd_produto.nomeresumido::text ~* 'PROENF' THEN 'Enfermagem'
                WHEN vd_produto.nomeresumido::text ~* 'PROFISIO' THEN 'Fisioterapia'
                WHEN vd_produto.nomeresumido::text = ANY(ARRAY['PRONEUROPSI','PROPSICO','PROCOGNITIVA']) THEN 'Psicologia'
                ELSE 'Outras Áreas' END
        END AS area,
        vd_compraitem.valor * vd_compra.valortotal / NULLIF(SUM(vd_compraitem.valor) OVER (PARTITION BY vd_compra.compraid), 0) AS value,
            CASE
        WHEN vd_compra."codVendedor" IS NULL THEN 'eCommerce'
        WHEN LEFT(vd_compra."codVendedor"::text, 1) = '8' THEN 'Representantes'
        WHEN vd_compra."codVendedor"::text IN ('9186', '9323', '9326', '9325') THEN 'Receptivo'
        WHEN LEFT(vd_compra."codVendedor"::text, 1) = 'R' THEN 'Renovacao'
        ELSE 'Call Center'
    END AS channel,
        CASE
        WHEN vd_compra."codVendedor" IS NULL THEN DATE(timezone('America/Sao_Paulo'::text, timezone('UTC'::text, vd_compra.datahora)))
        WHEN vd_compra."codVendedor"::text = '8000'::text THEN DATE(timezone('America/Sao_Paulo'::text, timezone('UTC'::text, vd_compra.datahora)))
        ELSE DATE(vd_compra.datahora)
        
    END AS data
    FROM app_sispag_pagamento.vd_compra
    LEFT JOIN app_sispag_pagamento.vd_compraitem ON vd_compra.compraid = vd_compraitem.compraid
    LEFT JOIN app_sispag_pagamento.vd_produto ON vd_compraitem.produtoid = vd_produto.produtoid
    LEFT JOIN app_sispag_pagamento.vd_request ON vd_compra.requestid = vd_request.requestid
    LEFT JOIN app_sispag_pagamento.vd_cliente ON vd_compra.clienteid = vd_cliente.clienteid
    LEFT JOIN bu_secad.programs ON vd_produto.nomeresumido::text = programs.program
    WHERE DATE(vd_compra.datahora AT TIME ZONE 'UTC' AT TIME ZONE 'America/Sao_Paulo') >= current_date - interval '5 months'
    AND vd_produto.tipoproduto::text IN ('P')
    AND vd_request.ambiente::text = 'P'
    AND LOWER(vd_cliente.nome::text) NOT LIKE '%teste%'
)
SELECT area,data,hr, COUNT(*) as qtd_vendas, SUM(value) as total_valor
FROM Sispag
WHERE channel IN ('Call Center','Receptivo')
GROUP BY area,data,hr
ORDER BY total_valor DESC
"""

QRY_OPERACIONAL = """
WITH ranked_calls AS (
    SELECT
        RIGHT(phone_number,11) AS phone,

        ROW_NUMBER() OVER (
            PARTITION BY RIGHT(phone_number,11)
            ORDER BY start_agent_date DESC
        ) AS rn,

        CASE
            WHEN tablename ~* 'mental|psicologia' THEN 'Psicologia'
            WHEN tablename ~* 'multi' THEN 'Multi'
            WHEN tablename ~* 'fisio' THEN 'Fisioterapia'
            WHEN tablename ~* 'enf' THEN 'Enfermagem'
            WHEN tablename ~* 'medic' THEN 'Medicina'
            WHEN tablename ~* 'nutri' THEN 'Nutrição'
            WHEN tablename ~* 'vet' THEN 'Veterinária'
            WHEN tablename ~* 'ped' THEN 'Pediatria'
            WHEN tablename ~* 'psiquia' THEN 'Psiquiatria'
            ELSE 'Outras Áreas'
        END AS area,

        CASE
            WHEN EXTRACT(EPOCH FROM wrap_duration) > 0 THEN 1
            ELSE 0
        END AS atendida,
        EXTRACT(EPOCH FROM agent_duration) AS call_duration,

        DATE(start_agent_date) AS data

    FROM integration_operations.vw_call_center_calls
    WHERE campaign_id IN (1025,1553,1605,1690,1299)
    AND start_agent_date::date >= current_date - interval '5 months'
)

SELECT
    area,
    data,
    ROUND(AVG(call_duration)/60,2) AS tma_min,
    COUNT(*) AS tentativas,
    SUM(atendida) AS atendidas
FROM ranked_calls
WHERE rn = 1
GROUP BY area, data

"""

# --- CARREGAMENTO ---
try:
    engine = get_engine()
    df_vendas = pd.read_sql(text(QRY_VENDAS), engine)
    df_vendas['data'] = pd.to_datetime(df_vendas['data'],errors='coerce')
    df_operacional = pd.read_sql(text(QRY_OPERACIONAL), engine)
    df_operacional['data'] = pd.to_datetime(df_operacional['data'], errors='coerce')
except Exception as e:
    st.error(f"Erro na conexão: {e}")
    df_vendas = pd.DataFrame()


In [ ]:
areas_disponiveis = df_vendas['area'].unique().tolist() if not df_vendas.empty else []
df_filtrado = df_vendas[df_vendas['area'].isin(areas_disponiveis)] if not df_vendas.empty else pd.DataFrame()
df_operacional_filtrado = df_operacional[df_operacional['area'].isin(areas_disponiveis)] if not df_operacional.empty else pd.DataFrame()

In [ ]:
hoje = pd.to_datetime(datetime.datetime.now().date())

df_fin = (
    df_operacional_filtrado.loc[
        (df_operacional_filtrado['data'] >= hoje - pd.DateOffset(months=5)) &
        (df_operacional_filtrado['data'] < hoje)
    ]
    .copy()
)

df_fin['mes'] = df_fin['data'].dt.to_period('M')
df_hist_fin_mensal =(
    df_fin
    .groupby('mes',as_index=False)
    .agg(
        tentativas=('tentativas','sum'),
        atendidas=('atendidas','sum')
    )
)
df_fin_hist =(
    df_filtrado.loc[
        (df_filtrado['data'] >= hoje - pd.DateOffset(months=5)) &
        (df_filtrado['data'] < hoje)
    ]
    .copy()
)
df_fin_hist['mes'] = df_fin_hist['data'].dt.to_period('M')
df_fin_mensal = (
    df_fin_hist.groupby('mes',as_index=False).agg(
        qtde_vendas = ('qtd_vendas','sum')
    )
)


In [49]:
df_fin_mensal = df_fin_mensal.merge(
    df_hist_fin_mensal[['mes','atendidas']],
    how='left',
    on='mes'
)


In [50]:
df_fin_mensal['tx_conv'] = (df_fin_mensal['qtde_vendas'] / df_fin_mensal['atendidas'] * 100)


In [51]:
conv_hist = df_fin_mensal['tx_conv'].mean()
conv_hist

np.float64(6.977583890765317)